In [188]:
import redis
import json
import datetime
import pandas as pd

class InitRedis:
    def __init__(self, host='127.0.0.1', port=6379, db=0):
        self.redis_pool = redis.ConnectionPool(host=host, port=port, db=db, decode_responses=True)
        self.redis_conn = redis.Redis(connection_pool=self.redis_pool)

    def set_data(self, key_name, value):
        self.redis_conn.set(key_name, value)

    def get_data(self, key_name):
        return self.redis_conn.get(key_name)

    def set_dict(self, key_name, dict_obj):
        self.redis_conn.set(key_name, json.dumps(dict_obj, ensure_ascii=False).encode('utf-8'))
    
    def get_dict(self, key_name):
        dumped_json = self.redis_conn.get(key_name)
        return dict(json.loads(dumped_json.decode('utf-8')))
        
    def hset_dict(self, keyname, dict_obj):
        self.redis_conn.hset(keyname, mapping=dict_obj)

    def hget_dict(self, keyname):
        return self.redis_conn.hgetall(keyname)

    def close_connector(self):
        self.redis_conn.close()

    def disconnect_pool(self):
        self.redis_pool.disconnect()

    def close_all(self):
        self.close_connector()
        self.disconnect_pool()

    def read_json_to_df(self):
        redis_key_list = self.redis_conn.scan(0)[1]
        redis_data_dict_list = []
        for each_key in redis_key_list:
            redis_data_dict_list.append(json.loads(self.get_data(each_key)))

        redis_data_df = pd.DataFrame(redis_data_dict_list)
        redis_data_df = redis_data_df.where(redis_data_df.notnull(), None)

        if 'datetime' in redis_data_df.columns:
            redis_data_df.loc[:, 'datetime'] = pd.to_datetime(redis_data_df.loc[:, 'datetime'])
        
        if 'datetime_end' in redis_data_df.columns:
            redis_data_df.loc[:, 'datetime_end'] = pd.to_datetime(redis_data_df.loc[:, 'datetime_end'])

        return redis_data_df

In [2]:
import pandas as pd
import os
import sys
sys.path.append("/home/coin_trading_bot/kimp_bot_core")
from etc.db_handler.create_schema_tables import InitDBClient

In [3]:
local_db_dict = {
    'host': 'localhost',
    'port': 3306,
    'user': 'root',
    'passwd': '13245dh!!',
    'database': 'test'
}

In [4]:
import time

test_client = InitDBClient(**local_db_dict)
sql = """
SELECT 
id,
user_id,
last_updated_timestamp,
redis_uuid,
symbol,
IFNULL(addcoin_uuid, '') as addcoin_uuid,
high,
low,
IFNULL(rolling_window, '') as rolling_window,
IFNULL(switch, '') as switch,
IFNULL(auto_trade_switch, '') as auto_trade_switch,
IFNULL(auto_trade_capital, '') as auto_trade_capital,
IFNULL(enter_upbit_uuid, '') as enter_upbit_uuid,
IFNULL(enter_binance_orderId, '') as enter_binance_orderId,
IFNULL(exit_upbit_uuid, '') as exit_upbit_uuid,
IFNULL(exit_binance_orderId, '') as exit_binance_orderId,
IFNULL(remark, '') as remark
FROM addcoin
"""
test_client.curr.execute(sql)
addcoin_dict_list = test_client.curr.fetchall()
addcoin_df = pd.DataFrame(addcoin_dict)
test_client.conn.close()

NameError: name 'addcoin_dict' is not defined

In [ ]:
addcoin_dict_list[0]

{'id': 4,
 'user_id': 1390695186,
 'last_updated_timestamp': 16625276521286262,
 'redis_uuid': '01ddbb4c-1c43-4da4-9c41-261388ef50bc',
 'symbol': 'BTC',
 'addcoin_uuid': '',
 'high': 7.0,
 'low': 4.0,
 'rolling_window': '',
 'switch': '0',
 'auto_trade_switch': '',
 'auto_trade_capital': '',
 'enter_upbit_uuid': '',
 'enter_binance_orderId': '',
 'exit_upbit_uuid': '',
 'exit_binance_orderId': '',
 'remark': ''}

In [163]:
redis_client_1 = InitRedis(db=1)
redis_client_2 = InitRedis(db=2)
redis_client_3 = InitRedis(db=3)

In [11]:
import time


In [107]:
test_client.curr.execute("""DESCRIBE user_info""")
fetched_list = test_client.curr.fetchall()
column_names = [x['Field'] for x in fetched_list if x['Field'] not in ['datetime', 'datetime_end']]

In [108]:
userinfo_table_name = 'user_info'

sql = "SELECT DATE_FORMAT(datetime, '%Y-%m-%d %T') as datetime, DATE_FORMAT(datetime_end, '%Y-%m-%d %T') as datetime_end"
for each_col in column_names:
    sql += ", " + each_col

sql += f" FROM {userinfo_table_name}"

test_client.curr.execute(sql)
test_client.curr.fetchall()

[{'datetime': '2022-09-07 11:45:52',
  'datetime_end': '2023-09-07 11:45:52',
  'user_id': 1390695186,
  'register_origin': 'kimp_bot1',
  'status': None,
  'user_name': 'charlie1155',
  'binance_uid': None,
  'referral_use': None,
  'referral_code': None,
  'interest_coin': None,
  'binance_leverage': 1,
  'binance_cross': 0,
  'binance_margin_call': 0,
  'safe_reverse': 0,
  'alarm_num': 1,
  'alarm_period': 1,
  'kimp_diff': None,
  'addcir_limit': None,
  'addcir_num_limit': None,
  'exd_percent': None,
  'kimp_diff_wallet': None,
  'kimp_diff_coin': None,
  'on_off': 1,
  'remark': None}]

In [194]:
def load_userinfo_to_redis(redis_client_1, db_dict, userinfo_table_name='user_info'):
    db_client = InitDBClient(**db_dict)
    # Fetch column names
    db_client.curr.execute(f"""DESCRIBE {userinfo_table_name}""")
    fetched_list = db_client.curr.fetchall()
    column_names = [x['Field'] for x in fetched_list if x['Field'] not in ['user_id', 'datetime', 'datetime_end']]

    sql = "SELECT user_id, DATE_FORMAT(datetime, '%Y-%m-%d %T') as datetime, DATE_FORMAT(datetime_end, '%Y-%m-%d %T') as datetime_end"
    for each_col in column_names:
        sql += ", " + each_col

    sql += f" FROM {userinfo_table_name}"
    
    db_client.curr.execute(sql)
    userinfo_dict_list = db_client.curr.fetchall()
    db_client.conn.close()

    if len(userinfo_dict_list) == 0:
        redis_client_1.redis_conn.flaushdb()
        return

    db_whole_key_list = []
    for each_userinfo in userinfo_dict_list:
        user_id = each_userinfo['user_id']
        db_whole_key_list.append(user_id)
        redis_client_1.set_data(user_id, json.dumps(each_userinfo))

    redis_whole_key_list = [int(x) for x in redis_client_1.redis_conn.scan(0)[1]]

    for not_existing_key in [x for x in redis_whole_key_list if x not in db_whole_key_list]:
        redis_client_1.redis_conn.delete(not_existing_key)

load_userinfo_to_redis(redis_client_1, local_db_dict, userinfo_table_name='user_info')
    

In [14]:
redis_client_2 = InitRedis(db=2)

In [114]:
def load_user_api_key_to_redis(redis_client_2, db_dict, encryption_key, user_api_key_table_name='user_api_key'):
    db_client = InitDBClient(**db_dict)
    # Fetch column names
    db_client.curr.execute(f"""DESCRIBE {user_api_key_table_name}""")
    fetched_list = db_client.curr.fetchall()
    column_names = [x['Field'] for x in fetched_list if x['Field'] not in ['id', 'user_id', 'datetime', 'access_key', 'secret_key']]

    sql = "SELECT id, user_id, DATE_FORMAT(datetime, '%Y-%m-%d %T') as datetime"
    sql += ", CONVERT(AES_DECRYPT(UNHEX(access_key), SHA2('{encryption_key}', 256)) USING UTF8) as access_key".format(encryption_key=encryption_key)
    sql += ", CONVERT(AES_DECRYPT(UNHEX(secret_key), SHA2('{encryption_key}', 256)) USING UTF8) as secret_key".format(encryption_key=encryption_key)
    for each_col in column_names:
        sql += ", " + each_col
    sql += f" FROM {user_api_key_table_name}"

    db_client.curr.execute(sql)
    user_api_key_dict_list = db_client.curr.fetchall()
    db_client.conn.close()

    if len(user_api_key_dict_list) == 0:
        redis_client_2.redis_conn.flaushdb()
        return

    db_whole_key_list = []
    for each_userinfo in user_api_key_dict_list:
        id = each_userinfo['id']
        db_whole_key_list.append(id)
        redis_client_2.set_data(id, json.dumps(each_userinfo))

    redis_whole_key_list = [int(x) for x in redis_client_2.redis_conn.scan(0)[1]]

    for not_existing_key in [x for x in redis_whole_key_list if x not in db_whole_key_list]:
        redis_client_2.redis_conn.delete(not_existing_key)

load_user_api_key_to_redis(redis_client_2, local_db_dict, '13245dh!!')

In [175]:
def load_addcoin_to_redis(redis_client_1, redis_client_3, db_dict, addcoin_table_name='addcoin', exclude_expired_user=True, debug=False):
    db_client = InitDBClient(**db_dict)
    # Fetch column names
    db_client.curr.execute(f"""DESCRIBE {addcoin_table_name}""")
    fetched_list = db_client.curr.fetchall()
    column_names = [x['Field'] for x in fetched_list if x['Field'] not in ['id', 'user_id', 'datetime']]

    sql = f"""SELECT id, user_id, DATE_FORMAT(datetime, '%Y-%m-%d %T') as datetime"""
    
    for each_col in column_names:
        sql += ", " + each_col
    sql += f" FROM {addcoin_table_name}"

    db_client.curr.execute(sql)
    db_addcoin_dict_list = db_client.curr.fetchall()
    db_client.conn.close()
    db_addcoin_df = pd.DataFrame(db_addcoin_dict_list)
    db_addcoin_df = db_addcoin_df.where(db_addcoin_df.notnull(), None)
    

    if len(db_addcoin_dict_list) == 0:
        redis_client_3.redis_conn.flaushdb()
        return

    if exclude_expired_user:
        # Load userinfo from redis db1
        redis_userinfo_df = read_data_from_redis(redis_client_1)
        
        db_addcoin_df = (db_addcoin_df[db_addcoin_df['user_id'].isin
        (redis_userinfo_df[redis_userinfo_df['datetime_end']>=datetime.datetime.now()]['user_id'].to_list())])

    for row_tup in db_addcoin_df.iterrows():
        redis_addcoin_df = read_data_from_redis(redis_client_3)
        index = row_tup[0]
        row = row_tup[1]
        if len(redis_addcoin_df) == 0 or row['redis_uuid'] not in redis_addcoin_df['redis_uuid'].to_list():
            if debug:
                print(f"addcoin|redis_uuid is not in Redis, appending a new row to Redis")
            redis_client_3.set_data(row['redis_uuid'], json.dumps(row.to_dict()))
        else:
            if debug:
                print(f"addcoin|redis_uuid already exists in Redis.")
            corresponding_row = redis_addcoin_df[redis_addcoin_df['redis_uuid']==row['redis_uuid']]
            existing_timestamp = corresponding_row['last_updated_timestamp'].values[0]
            if row['last_updated_timestamp'] > existing_timestamp:
                if debug:
                    print("addcoin|newer addcoin setting detected. updating the existing row in Redis..")
                redis_client_3.set_data(row['redis_uuid'], json.dumps(row.to_dict()))

    redis_addcoin_df = read_data_from_redis(redis_client_3)
    for row_tup in redis_addcoin_df.iterrows():
        index = row_tup[0]
        row = row_tup[1]
        if row['redis_uuid'] not in db_addcoin_df['redis_uuid'].to_list():
            if debug:
                print(f"addcoin|redis_uuid doesn't exist in the db, removing row that has a uuid of {row['redis_uuid']} from Redis..")
            redis_client_3.redis_conn.delete(row['redis_uuid'])

load_addcoin_to_redis(redis_client_1, redis_client_3, local_db_dict, addcoin_table_name='addcoin', exclude_expired_user=True, debug=True)
read_data_from_redis(redis_client_3)

addcoin|redis_uuid already exists in Redis.
addcoin|redis_uuid already exists in Redis.
addcoin|redis_uuid already exists in Redis.
addcoin|redis_uuid already exists in Redis.
addcoin|redis_uuid doesn't exist in the db, removing row that has a uuid of 4d95891c-1a91-4e26-bf04-20f05b5b6695 from Redis..


,id,user_id,datetime,last_updated_timestamp,redis_uuid,symbol,addcoin_uuid,high,low,rolling_window,switch,auto_trade_switch,auto_trade_capital,enter_upbit_uuid,enter_binance_orderId,exit_upbit_uuid,exit_binance_orderId,remark
0,6,1390695186,2022-09-07 14:18:16,16625277732928040,ea11ec6c-fc97-4319-a99e-44762d2113d3,EOS,None,0.0,-3.0,None,None,None,None,None,None,None,None,None
1,4,1390695186,2022-09-08 14:37:51,16625276521286272,01ddbb4c-1c43-4da4-9c41-261388ef50bc,BTC,None,7.0,4.0,None,None,None,None,None,None,None,None,None
2,5,1390695186,2022-09-08 14:39:00,16625276521304791,cda323c5-d874-40fe-a2d3-fe088a20778e,XRP,None,8.0,4.0,None,None,None,None,None,None,None,None,None
3,8,1390695186,2022-09-07 14:18:16,16625278489318240,97b39296-4c2d-431d-90aa-21f577a76ac9,ETHUSDT,None,1900.0,1800.0,None,None,None,None,None,None,None,None,None


In [190]:
redis_client_4 = InitRedis(db=4)

In [201]:
def load_addcir_to_redis(redis_client_1, redis_client_4, db_dict, addcir_table_name='addcir', exclude_expired_user=True, debug=False):
    db_client = InitDBClient(**db_dict)
    # Fetch column names
    db_client.curr.execute(f"""DESCRIBE {addcir_table_name}""")
    fetched_list = db_client.curr.fetchall()
    column_names = [x['Field'] for x in fetched_list if x['Field'] not in ['id', 'user_id', 'datetime']]

    sql = f"""SELECT id, user_id, DATE_FORMAT(datetime, '%Y-%m-%d %T') as datetime"""
    
    for each_col in column_names:
        sql += ", " + each_col
    sql += f" FROM {addcir_table_name}"

    db_client.curr.execute(sql)
    db_addcir_dict_list = db_client.curr.fetchall()
    db_client.conn.close()
    db_addcir_df = pd.DataFrame(db_addcir_dict_list)
    db_addcir_df = db_addcir_df.where(db_addcir_df.notnull(), None)
    

    if len(db_addcir_dict_list) == 0:
        redis_client_4.redis_conn.flushdb()
        return

    if exclude_expired_user:
        # Load userinfo from redis db1
        redis_userinfo_df = redis_client_1.read_json_to_df()
        
        db_addcir_df = (db_addcir_df[db_addcir_df['user_id'].isin
        (redis_userinfo_df[redis_userinfo_df['datetime_end']>=datetime.datetime.now()]['user_id'].to_list())])

    for row_tup in db_addcir_df.iterrows():
        redis_addcir_df = redis_client_4.read_json_to_df()
        index = row_tup[0]
        row = row_tup[1]
        if len(redis_addcir_df) == 0 or row['redis_uuid'] not in redis_addcir_df['redis_uuid'].to_list():
            if debug:
                print(f"addcir|redis_uuid is not in Redis, appending a new row to Redis")
            redis_client_4.set_data(row['redis_uuid'], json.dumps(row.to_dict()))
        else:
            if debug:
                print(f"addcir|redis_uuid already exists in Redis.")
            corresponding_row = redis_addcir_df[redis_addcir_df['redis_uuid']==row['redis_uuid']]
            existing_timestamp = corresponding_row['last_updated_timestamp'].values[0]
            if row['last_updated_timestamp'] > existing_timestamp:
                if debug:
                    print("addcir|newer addcir setting detected. updating the existing row in Redis..")
                redis_client_4.set_data(row['redis_uuid'], json.dumps(row.to_dict()))

    redis_addcir_df = redis_client_4.read_json_to_df()
    for row_tup in redis_addcir_df.iterrows():
        index = row_tup[0]
        row = row_tup[1]
        if row['redis_uuid'] not in db_addcir_df['redis_uuid'].to_list():
            if debug:
                print(f"addcir|redis_uuid doesn't exist in the db, removing row that has a uuid of {row['redis_uuid']} from Redis..")
            redis_client_4.redis_conn.delete(row['redis_uuid'])

load_addcir_to_redis(redis_client_1, redis_client_4, local_db_dict, addcir_table_name='addcir', exclude_expired_user=True, debug=True)
redis_client_4.read_json_to_df()

KeyError: 'datetime_end'